# Setup

In [1]:
from torch.cuda import is_available as is_cuda_available
from numpy import nan
from lighthouse.models import CGDETRPredictor
from utils.helper_methods import split_video_ffmpeg, parse_timestamps, extract_clips, adjust_video_speed
import pandas
import os

C:\Users\denve\miniconda3\envs\moment-retrieval-and-video-summarization\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model weights can be downloaded from https://zenodo.org/records/13960580
Checkpoint file should correspond to the feature selected in predictor

For slowfast video encoding features additional weights should be downloaded. They can be found in lighthouse GitHub page (https://github.com/line/lighthouse/tree/main) or downloaded from [here](https://dl.fbaipublicfiles.com/pyslowfast/model_zoo/kinetics400/SLOWFAST_8x8_R50.pkl)

In [2]:
device = "cuda" if is_cuda_available() else "cpu"
device

'cuda'

In [3]:
# model = CGDETRPredictor('model_weights/clip_cg_detr_qvhighlight.ckpt', device=device,
#                         feature_name='clip')

model = CGDETRPredictor('model_weights/clip_slowfast_cg_detr_qvhighlight.ckpt', device=device,
                        feature_name='clip_slowfast', slowfast_path="model_weights/SLOWFAST_8x8_R50.pkl", pann_path=None)

# Video preparation
CG-DETR model has a major limitation -- it doesn't allow videos longer than 150 seconds. We will try several approaches to this problem

In [7]:
# Load events which should be retrieved
events = pandas.read_csv("Annotations/7-TVSum-video_25-i6324060.txt")[['annotation', 'timestamp']]
events['timestamp'].replace(nan, None)
events['timestamp'] = events['timestamp'].map(parse_timestamps)

# set the original video path
og_video_path = "TVSum/video_25.mp4"

## Splitting
The most obvious way is splitting videos in chunks. We cut the original video in segments no longer than 150 seconds and then process them seperately

In [8]:
output_path = "processed_videos/video_25_chunks"
split_video_ffmpeg(og_video_path, output_path, 150)

Splitting TVSum/video_25.mp4 into 150s chunks...
Chunks are successfully saved into 'processed_videos/video_25_chunks' folder.


'processed_videos/video_25_chunks'

In [9]:
results = []

for row in events.iterrows():
    query, timestamp = row[1]
    predicted_timestamps = []
    for index, chunk in enumerate(os.listdir(output_path)):
        if chunk.endswith('.mp4'):
            video_path = output_path + '/' + chunk
            video = model.encode_video(video_path)
            predicted_timestamps.append((index, model.predict(query, video)))

    results.append({
        "query": query,
        "true_timestamp": timestamp,
        "predictions": predicted_timestamps
    })

In [10]:
for query_result in results:
    print(f"Query: {query_result['query']}")
    print(f"True timestamp: {query_result['true_timestamp']}")


    for chunk in query_result['predictions']:
        print("\tChunk:", chunk[0])
        print(f"\t\tClosest Moment: {chunk[1]['pred_relevant_windows'][0][0], chunk[1]['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {chunk[1]['pred_relevant_windows'][0][2]}")
    print("-" * 50)

Query: Two young men are traveling on a train and recording each other with cameras
True timestamp: (3.0, 6.0)
	Chunk: 0
		Closest Moment: (2.6967, 8.1025), Corresponding Confidence score: 0.9879
	Chunk: 1
		Closest Moment: (87.9291, 100.7757), Corresponding Confidence score: 0.9994
--------------------------------------------------
Query: Short clips with landscape views from a train window are shown
True timestamp: (7.0, 12.0)
	Chunk: 0
		Closest Moment: (132.779, 144.7218), Corresponding Confidence score: 0.9916
	Chunk: 1
		Closest Moment: (0.0, 8.5486), Corresponding Confidence score: 0.9993
--------------------------------------------------
Query: A group of young people are warming up together
True timestamp: (27.0, 37.0)
	Chunk: 0
		Closest Moment: (24.9719, 54.2116), Corresponding Confidence score: 0.9969
	Chunk: 1
		Closest Moment: (23.978, 50.8973), Corresponding Confidence score: 0.9931
--------------------------------------------------
Query: A young man performs parkour ma

Now we have a problem, because we get a timestamp for each chunk, but we only need one timestamp for the whole video. The simplest solution is to pick confidence score, although this approach has a lot of shortcomings, we will use it, as alternatives require more complex algorithms

In [11]:
chunks_timestamp_results = []
for query_result in results:
    max_score = 0.0
    best_timestamp = ()
    for chunk in query_result['predictions']:
        prediction = chunk[1]
        if prediction['pred_relevant_windows'][0][2] > max_score:
            max_score = prediction['pred_relevant_windows'][0][2]
            best_timestamp = (prediction['pred_relevant_windows'][0][0], prediction['pred_relevant_windows'][0][1])

    chunks_timestamp_results.append(best_timestamp)


## Speeding up
Another approach is speeding up the video until it becomes short enough

In [12]:
output_path = "processed_videos/video_25_sped_up.mp4"
adjust_video_speed(og_video_path, output_path, 150)

Speeding up video by 1.83x...
Video is successfully sped up


In [13]:
results = []

for row in events.iterrows():
    query, timestamp = row[1]
    predicted_timestamps = []

    video = model.encode_video(output_path)
    prediction = model.predict(query, video)

    results.append({
        "query": query,
        "true_timestamp": timestamp,
        "predictions": prediction
    })

In [14]:
for query_result in results:
    print(f"Query: {query_result['query']}")
    print(f"True timestamp: {query_result['true_timestamp']}")

    prediction = query_result['predictions']
    print(f"\t\tClosest Moment: {prediction['pred_relevant_windows'][0][0], prediction['pred_relevant_windows'][0][1]}, Corresponding Confidence score: {prediction['pred_relevant_windows'][0][2]}")
    print("-" * 50)

Query: Two young men are traveling on a train and recording each other with cameras
True timestamp: (3.0, 6.0)
		Closest Moment: (1.2526, 7.1122), Corresponding Confidence score: 0.9907
--------------------------------------------------
Query: Short clips with landscape views from a train window are shown
True timestamp: (7.0, 12.0)
		Closest Moment: (82.3009, 92.9174), Corresponding Confidence score: 0.985
--------------------------------------------------
Query: A group of young people are warming up together
True timestamp: (27.0, 37.0)
		Closest Moment: (98.2615, 112.8362), Corresponding Confidence score: 0.9865
--------------------------------------------------
Query: A young man performs parkour maneuvers over obstacles
True timestamp: (44.0, 46.0)
		Closest Moment: (102.2807, 112.4543), Corresponding Confidence score: 0.9848
--------------------------------------------------
Query: A young man films a company of young people on a camera
True timestamp: (52.0, 56.0)
		Closest Mom

In [15]:
sped_up_timestamps_results = []

for query_result in results:
    sped_up_timestamps_results.append(query_result['predictions'])